# บทที่ 11: พื้นฐาน SQL สำหรับเชื่อมข้อมูลเพื่อการวิเคราะห์

ในการวิเคราะห์ข้อมูลจริง ข้อมูลที่ต้องใช้มักไม่ได้อยู่ในตารางเดียว 
แต่อาจกระจายอยู่หลายชุดข้อมูล เช่น ข้อมูลจากหลายหน่วยงาน หลายระบบ หรือหลายไฟล์ 
ในบทนี้ เราจะเรียนรู้การใช้ SQL เบื้องต้นผ่าน DuckDB บน Python 
เพื่อเชื่อมข้อมูลหลายชุดเข้าด้วยกัน และสร้างผลลัพธ์ที่นำไปใช้วิเคราะห์ต่อได้

## ผลการเรียนรู้ที่คาดหวัง

เมื่อจบบทเรียนนี้ ผู้เรียนจะสามารถ: 
1. นำเข้าข้อมูลจากไฟล์ Excel และ CSV มาเป็น pandas DataFrame ได้ 
2. ใช้ DuckDB เพื่อ query ข้อมูลจาก pandas DataFrame ได้ 
3. เขียน SQL เบื้องต้นได้ เช่น `SELECT`, `WHERE`, `ORDER BY`, `GROUP BY` 
4. ใช้ aggregate function เช่น `COUNT`, `COUNT DISTINCT`, `SUM`, `AVG`, `MIN`, `MAX` 
5. อธิบายความแตกต่างของ `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, และ `FULL OUTER JOIN` ได้ 
6. เลือกชนิดของ JOIN ให้เหมาะกับคำถามการวิเคราะห์ได้ 
7. เชื่อมข้อมูลจากหลายแหล่งในระดับพื้นที่ เช่น จังหวัด อำเภอ ตำบล ได้ 
8. ตรวจสอบผลลัพธ์หลัง JOIN ได้ เช่น ข้อมูลที่เชื่อมเจอ เชื่อมไม่เจอ หรือเกิดค่า NULL 
9. ใช้ `WITH` หรือ CTE เพื่อจัด query ให้อ่านง่ายขึ้น 
10. เก็บ SQL query เป็น Python function เพื่อเรียกใช้งานซ้ำได้

## ลำดับเนื้อหา 
บทนี้ประกอบด้วยหัวข้อดังนี้
1. แนวคิดการใช้ SQL เพื่อเชื่อมข้อมูล 
2. การนำเข้าข้อมูลตัวอย่าง 
3. การเตรียม DuckDB และ register DataFrame 
4. SQL พื้นฐาน: `SELECT`, `WHERE`, `ORDER BY` 
5. การสรุปข้อมูลด้วย `GROUP BY` 
6. การใช้ aggregate function 
7. แนวคิดของ JOIN และผลลัพธ์ที่เกิดจาก JOIN แต่ละแบบ 
8. การเชื่อมข้อมูลระดับพื้นที่ 
9. การใช้ `COALESCE` เพื่อจัดการค่า NULL หลัง JOIN 
10. การตรวจสอบผลลัพธ์หลัง JOIN 
11. การเก็บ query เป็น Python function 
12. แบบฝึกหัดท้ายบท

## 3.1 แนวคิดการใช้ SQL เพื่อเชื่อมข้อมูล

SQL เป็นภาษาที่ใช้ทำงานกับข้อมูลในรูปแบบตาราง 

ในงานวิเคราะห์ เรามักใช้ SQL เพื่อตอบคำถาม เช่น 
- ข้อมูลแต่ละชุดมีจำนวน record เท่าไร 
- จังหวัดใดมีจำนวนเกษตรกรเปราะบางมากที่สุด 
- จังหวัดใดมีจำนวน logbook จาก MSO มากที่สุด 
- พื้นที่ใดมีข้อมูลจากทั้งสองแหล่ง 
- พื้นที่ใดมีข้อมูลเกษตรกรจำนวนมาก แต่ยังไม่มีข้อมูลจาก MSO 
- พื้นที่ใดควรนำไปวิเคราะห์เชิงลึกต่อ 

ในบทนี้ เราจะใช้ SQL ผ่าน DuckDB โดยให้ DuckDB query ข้อมูลที่อยู่ใน pandas DataFrame 

## 3.2 เตรียม Library

In [2]:
%pip install duckdb


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd 
import duckdb

## 3.3 นำเข้าข้อมูลตัวอย่าง 
ในบทนี้จะนำเข้าข้อมูล 2 ชุด 
1. ข้อมูลเกษตรกรเปราะบางจากไฟล์ Excel 
2. ข้อมูล logbook จากไฟล์ CSV 

หลังจากอ่านไฟล์แล้ว เราจะเก็บข้อมูลไว้ใน DataFrame ชื่อ 
- `farmer_df` 
- `mso_df`

In [4]:
farmer_path = "moac_opsmoac_fragile_farmer.xlsx" 
mso_path = "msdhs_ops_mso_logbook.csv"

In [5]:
# farmer_df = pd.read_excel(farmer_path) 
# farmer_df.head()

อย่างไรก็ตาม ในการใช้งานจริง ไฟล์ข้อมูลอาจไม่ได้อยู่ใน folder เดียวกับ notebook เสมอไป ดังนั้นก่อนอ่านไฟล์ ควรตรวจสอบตำแหน่งปัจจุบันของ notebook และตรวจสอบว่าไฟล์ข้อมูลอยู่ที่ path ใด หากระบุ path ไม่ถูกต้อง เช่น ใช้แค่ชื่อไฟล์โดยตรง แต่ไฟล์อยู่คนละ directory จะเกิด error เช่น

```text 
FileNotFoundError: [Errno 2] No such file or directory
```

ดังนั้นขั้นตอนที่ดีคือ
1. ตรวจสอบ current working directory
2. กำหนด folder ที่เก็บข้อมูล
3. ประกอบ path ของไฟล์ด้วย pathlib.Path
4. ตรวจสอบว่าไฟล์มีอยู่จริงก่อนอ่าน

In [6]:
from pathlib import Path

ตรวจสอบว่า notebook กำลังทำงานอยู่ที่ directory ใด

In [7]:
current_dir = Path.cwd() 
current_dir

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day5')

กำหนด directory ที่เก็บข้อมูล

ตัวอย่างนี้ไฟล์ข้อมูลอยู่ไม่ได้อยู่ใน folder เดียวกับ notebook

โครงสร้าง folder ตัวอย่าง:

```text
course/
├── day5/
│   └── basic_sql.ipynb
└── day3/
    ├── moac_opsmoac_fragile_farmer.xlsx
    └── msdhs_ops_mso_logbook.csv
```

ถ้า notebook อยู่ใน folder notebooks และข้อมูลอยู่ใน folder data ที่อยู่ข้างนอก notebook folder
จะต้องใช้ path `/workspaces/MSDHS_OJT/py-jupyter_docker/course`

In [8]:
data_dir = Path("/workspaces/MSDHS_OJT/py-jupyter_docker/course") 
data_dir

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course')

กำหนด path ของไฟล์ข้อมูลแต่ละไฟล์

In [9]:
farmer_path = data_dir / "day3/moac_opsmoac_fragile_farmer.xlsx" 
mso_path = data_dir / "day3/msdhs_ops_mso_logbook.csv" 

In [10]:
farmer_path

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day3/moac_opsmoac_fragile_farmer.xlsx')

In [11]:
mso_path

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day3/msdhs_ops_mso_logbook.csv')

ตรวจสอบว่าไฟล์มีอยู่จริงหรือไม่ก่อนอ่านข้อมูล 
ถ้าผลลัพธ์เป็น `False` แปลว่า path ยังไม่ถูกต้อง ต้องกลับไปตรวจสอบว่า 
- notebook อยู่ที่ directory ใด 
- ไฟล์ข้อมูลอยู่ที่ directory ใด 
- ชื่อไฟล์สะกดถูกต้องหรือไม่ 
- นามสกุลไฟล์ถูกต้องหรือไม่ เช่น `.xlsx`, `.csv`

In [12]:
print("Farmer file exists:", farmer_path.exists())

Farmer file exists: True


In [13]:
print("MSO file exists:", mso_path.exists())

MSO file exists: True


ถ้าต้องการดูไฟล์ทั้งหมดใน folder ข้อมูล สามารถใช้คำสั่งนี้เพื่อตรวจสอบชื่อไฟล์

In [14]:
list(data_dir.iterdir())

[PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day3'),
 PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day5'),
 PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day2')]

หลังจากตรวจสอบแล้วว่า path ถูกต้อง จึงอ่านไฟล์เข้ามาเป็น DataFrame

In [15]:
mso_usecols = [ 
    "วันที่แก้ไขข้อมูลล่าสุด", 
    "รหัสครัวเรือน", 
    "รหัสประจำบ้าน", 
    "วันที่สร้างครัวเรือน",
    "ตำบล/แขวง",
    "เขต/อำเภอ/เทศบาล", 
    "จังหวัด", 
    "อายุ", 
    "เพศ", 
    "ระดับการศึกษา", 
    "อาชีพหลัก", 
    "ประเภทกลุ่มเป้าหมาย", 
    "รหัส cm", 
    "หน่วยงานของ cm" 
] 

mso_dtype = { 
    "รหัสครัวเรือน": "string", 
    "รหัสประจำบ้าน": "string", 
    "รหัส cm": "string" } 
    
mso_df = pd.read_csv( 
    mso_path, 
    encoding="utf-8-sig", 
    usecols=mso_usecols, 
    dtype=mso_dtype, 
    parse_dates=["วันที่แก้ไขข้อมูลล่าสุด", "วันที่สร้างครัวเรือน"] 
) 

mso_df

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,ตำบล/แขวง,เขต/อำเภอ/เทศบาล,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,โนนทอง,นายูง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ท่านัด,ดำเนินสะดวก,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,นาข่า,ท่าบ่อ,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,โดมประดิษฐ์,น้ำยืน,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,หนองหญ้าขาว,สีคิ้ว,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2025-07-29 14:33:32.178,688878eaa933a405b21cc136,<NA>,2025-07-29 14:31:54.610,59,NaN,โนนกอก,เกษตรสมบูรณ์,ชัยภูมิ,NaN,เกษตรกรรม (พืช ปศุสัตว์ ประมง),วัยผู้ใหญ่/วัยแรงงาน,cm360060,ศูนย์คุ้มครองคนไร้ที่พึ่ง จ.ชัยภูมิ
96,2023-03-17 19:08:15.489,628b008c8fa67bf5ad7d221b,<NA>,2022-05-23 10:33:32.270,79,NaN,เมืองเตา,พยัคฆภูมิพิสัย,มหาสารคาม,NaN,NaN,ผู้สูงอายุ,cm440005,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
97,2023-05-01 09:19:26.395,63b6325b2617b2b7aa04dc0b,25120251099,2023-05-01 09:13:47.520,51,หญิง,หนองหมากฝ้าย,วัฒนานคร,สระแก้ว,ประถมศึกษา,รับจ้างทั่วไป,วัยผู้ใหญ่/วัยแรงงาน,cm270001,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
98,2025-01-04 14:05:25.955,63e1d0d92617b2b7aa06096d,40160104483,2023-07-02 11:17:29.369,81,ชาย,หนองกุงธนสาร,ภูเวียง,ขอนแก่น,ไม่ได้เรียนหนังสือ,NaN,ผู้สูงอายุ,cm400016,ศูนย์เรียนรู้การพัฒนาสตรีและครอบครัวรัตนาภา จั...


In [16]:
target_sheets = [ 
    "v_cpd_fragile", 
    "v_dld_fragile", 
    "v_doae_fragile" 
] 

farmer_usecols = [ 
    "department_code", 
    "department", 
    "pid", 
    "province_code", 
    "province", 
    "amphur", 
    "tambon", 
    "is_farmer", 
    "farmer_type", 
    "main_occupation", 
    "income_in", 
    "income_out", 
    "debts_in", 
    "debts_out", 
    "updated_at" 
] 

farmer_dtype = { 
    "pid": "string", 
    "province_code": "string" 
} 

farmer_df_list = [] 

for sheet in target_sheets: 
    temp_df = pd.read_excel( 
        farmer_path, 
        sheet_name=sheet, 
        usecols=farmer_usecols, 
        dtype=farmer_dtype 
    ) 
    
    temp_df["source_sheet"] = sheet 
    
    farmer_df_list.append(temp_df) 
    
farmer_raw_df = pd.concat(farmer_df_list, ignore_index=True) 

farmer_raw_df

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,doae,กรมส่งเสริมการเกษตร,b437149cc5b6a79f8866914fd84b644196ed99ba4f3d62...,96,NaN,NaN,NaN,1,เกษตรกรด้านพืช,ประกอบการเกษตร,200000.0,0.0,0.0,0.0,20250825.0,v_doae_fragile
91,doae,กรมส่งเสริมการเกษตร,b1ce3ba83cfcfc5c21414fe2c496a37b635f56cb4de98c...,61,NaN,NaN,NaN,1,เกษตรกรด้านพืช,NaN,0.0,0.0,0.0,0.0,20250825.0,v_doae_fragile
92,doae,กรมส่งเสริมการเกษตร,5bdabb8ac705d3ba6fab31c9f7f1a48c37455ac548d72a...,53,NaN,NaN,NaN,1,เกษตรกรด้านพืช,ประกอบการเกษตร,200000.0,0.0,100000.0,0.0,20250825.0,v_doae_fragile
93,doae,กรมส่งเสริมการเกษตร,d066b479f17041cb1297e115b3ed0c8e8b724769855a97...,12,NaN,NaN,NaN,1,เกษตรกรด้านพืช,NaN,60000.0,0.0,0.0,0.0,20250825.0,v_doae_fragile


## 3.4 ตรวจสอบข้อมูลหลังนำเข้า 
หลังจากนำเข้าข้อมูลแล้ว เราจะมี DataFrame หลัก 2 ชุด 
- `farmer_raw_df` คือข้อมูลเกษตรกรเปราะบางจากหลาย sheet ในไฟล์ Excel 
- `mso_df` คือข้อมูล logbook จาก MSO 

ก่อนนำข้อมูลไป query ควรตรวจสอบข้อมูลเบื้องต้นก่อน เช่น 
- จำนวนแถวและจำนวนคอลัมน์ 
- ชื่อ column 
- ชนิดข้อมูล 
- ตัวอย่างข้อมูล 
- ค่า missing ใน column สำคัญ 

เพื่อตรวจสอบว่า DataFrame พร้อมสำหรับ JOIN ด้วย SQL หรือไม่

In [17]:
farmer_raw_df.shape

(95, 16)

In [18]:
farmer_raw_df.head()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile


In [19]:
farmer_raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   department_code  95 non-null     str    
 1   department       95 non-null     str    
 2   pid              95 non-null     string 
 3   province_code    67 non-null     string 
 4   province         10 non-null     object 
 5   amphur           10 non-null     object 
 6   tambon           10 non-null     object 
 7   is_farmer        95 non-null     int64  
 8   farmer_type      95 non-null     str    
 9   main_occupation  21 non-null     object 
 10  income_in        64 non-null     float64
 11  income_out       64 non-null     float64
 12  debts_in         64 non-null     float64
 13  debts_out        64 non-null     float64
 14  updated_at       80 non-null     float64
 15  source_sheet     95 non-null     str    
dtypes: float64(5), int64(1), object(4), str(4), string(2)
memory usage: 12.0+ K

In [20]:
mso_df.shape

(100, 14)

In [21]:
mso_df.head()

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,ตำบล/แขวง,เขต/อำเภอ/เทศบาล,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,โนนทอง,นายูง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ท่านัด,ดำเนินสะดวก,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,นาข่า,ท่าบ่อ,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,โดมประดิษฐ์,น้ำยืน,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,หนองหญ้าขาว,สีคิ้ว,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


In [22]:
mso_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   วันที่แก้ไขข้อมูลล่าสุด  100 non-null    datetime64[us]
 1   รหัสครัวเรือน            100 non-null    string        
 2   รหัสประจำบ้าน            24 non-null     string        
 3   วันที่สร้างครัวเรือน     100 non-null    datetime64[us]
 4   อายุ                     100 non-null    int64         
 5   เพศ                      77 non-null     str           
 6   ตำบล/แขวง                100 non-null    str           
 7   เขต/อำเภอ/เทศบาล         100 non-null    str           
 8   จังหวัด                  100 non-null    str           
 9   ระดับการศึกษา            77 non-null     str           
 10  อาชีพหลัก                39 non-null     str           
 11  ประเภทกลุ่มเป้าหมาย      100 non-null    str           
 12  รหัส cm                  100 non-null    string 

จากข้อมูลที่นำเข้า มี column สำคัญที่เกี่ยวข้องกับการเชื่อมข้อมูลดังนี้ 

### ข้อมูลเกษตรกร

| column | ความหมาย | 
|---|---| 
| department_code | รหัสหน่วยงาน | 
| department | หน่วยงาน | 
| pid | รหัสบุคคล | 
| province_code | รหัสจังหวัด | 
| province | จังหวัด | 
| amphur | อำเภอ | 
| tambon | ตำบล | 
| is_farmer | สถานะว่าเป็นเกษตรกรหรือไม่ | 
| farmer_type | ประเภทเกษตรกร | 
| main_occupation | อาชีพหลัก | 
| income_in | รายได้ในภาคเกษตร | 
| income_out | รายได้นอกภาคเกษตร | 
| debts_in | หนี้สินในภาคเกษตร | 
| debts_out | หนี้สินนอกภาคเกษตร | 
| updated_at | วันที่ปรับปรุงข้อมูล | 
| source_sheet | sheet ต้นทางของข้อมูล |

### ข้อมูล MSO 
| column | ความหมาย | 
|---|---| 
| วันที่แก้ไขข้อมูลล่าสุด | วันที่แก้ไขข้อมูลล่าสุด | 
| รหัสครัวเรือน | รหัสครัวเรือน | 
| รหัสประจำบ้าน | รหัสประจำบ้าน | 
| วันที่สร้างครัวเรือน | วันที่สร้างครัวเรือน | 
| ตำบล/แขวง | ตำบล | 
| เขต/อำเภอ/เทศบาล | อำเภอ | 
| จังหวัด | จังหวัด | 
| อายุ | อายุ | 
| เพศ | เพศ | 
| ระดับการศึกษา | ระดับการศึกษา | 
| อาชีพหลัก | อาชีพหลัก | 
| ประเภทกลุ่มเป้าหมาย | ประเภทกลุ่มเป้าหมาย | 
| รหัส cm | รหัส case manager | 
| หน่วยงานของ cm | หน่วยงานของ case manager |

## 3.5 เตรียมข้อมูลสำหรับใช้ Query 
เพื่อให้เขียน SQL ได้ง่ายขึ้น เราจะเตรียม DataFrame สำหรับใช้ในบทนี้ โดย 
1. คัดลอก `farmer_raw_df` มาเป็น `farmer_df` 
2. เปลี่ยนชื่อ column พื้นที่ของ `mso_df` ให้ตรงกับฝั่งเกษตรกร 
3. ทำความสะอาด key ที่จะใช้เชื่อมข้อมูล เช่น จังหวัด อำเภอ ตำบล 
4. สร้าง column เพิ่มเติมที่ช่วยให้การวิเคราะห์ง่ายขึ้น 

เหตุผลที่ต้องทำให้ชื่อ column พื้นที่ตรงกัน เพราะในขั้นตอน JOIN เราจะใช้พื้นที่เป็น key หลัก เช่น 
- `province` 
- `amphur` 
- `tambon` 

ถ้าชื่อ column ไม่ตรงกัน query จะอ่านยากและมีโอกาสเขียนผิดง่าย

In [23]:
farmer_df = farmer_raw_df.copy()

mso_df = mso_df.rename(columns={ 
    "จังหวัด": "province", 
    "เขต/อำเภอ/เทศบาล": "amphur", 
    "ตำบล/แขวง": "tambon", 
    "วันที่แก้ไขข้อมูลล่าสุด": "last_updated_at", 
    "วันที่สร้างครัวเรือน": "household_created_at", 
    "รหัสครัวเรือน": "household_id", 
    "รหัสประจำบ้าน": "house_code", 
    "อายุ": "age", 
    "เพศ": "gender", 
    "ระดับการศึกษา": "education_level", 
    "อาชีพหลัก": "main_occupation", 
    "ประเภทกลุ่มเป้าหมาย": "target_group", 
    "รหัส cm": "cm_id", 
    "หน่วยงานของ cm": "cm_department" 
})

In [24]:
list(farmer_df.columns)

['department_code',
 'department',
 'pid',
 'province_code',
 'province',
 'amphur',
 'tambon',
 'is_farmer',
 'farmer_type',
 'main_occupation',
 'income_in',
 'income_out',
 'debts_in',
 'debts_out',
 'updated_at',
 'source_sheet']

In [25]:
list(mso_df.columns)

['last_updated_at',
 'household_id',
 'house_code',
 'household_created_at',
 'age',
 'gender',
 'tambon',
 'amphur',
 'province',
 'education_level',
 'main_occupation',
 'target_group',
 'cm_id',
 'cm_department']

ตรวจสอบ column พื้นที่หลังปรับชื่อ

In [26]:
mso_df[["province", "amphur", "tambon"]]

,province,amphur,tambon
0,อุดรธานี,นายูง,โนนทอง
1,ราชบุรี,ดำเนินสะดวก,ท่านัด
2,หนองคาย,ท่าบ่อ,นาข่า
3,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์
4,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว
...,...,...,...
95,ชัยภูมิ,เกษตรสมบูรณ์,โนนกอก
96,มหาสารคาม,พยัคฆภูมิพิสัย,เมืองเตา
97,สระแก้ว,วัฒนานคร,หนองหมากฝ้าย
98,ขอนแก่น,ภูเวียง,หนองกุงธนสาร


In [27]:
farmer_df[["province", "amphur", "tambon"]]

,province,amphur,tambon
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
...,...,...,...
90,NaN,NaN,NaN
91,NaN,NaN,NaN
92,NaN,NaN,NaN
93,NaN,NaN,NaN


## 3.6 ทำความสะอาด Key สำหรับการ JOIN 
ในบทนี้ เราจะใช้ชื่อพื้นที่เป็น key สำหรับเชื่อมข้อมูล ได้แก่ 
- `province` 
- `amphur` 
- `tambon` 

เนื่องจาก key เหล่านี้เป็นข้อความ จึงอาจมีปัญหา เช่น 
- มีช่องว่างหัวท้าย 
- มีค่าขาดหาย 
- ใช้คำสะกดไม่ตรงกัน 
- มีชื่ออำเภอหรือตำบลในรูปแบบที่ต่างกัน 

ก่อน JOIN จึงควรทำความสะอาดเบื้องต้น โดยแปลงเป็น string และตัดช่องว่างหัวท้าย

In [28]:
area_cols = ["province", "amphur", "tambon"] 

for col in area_cols: 
    farmer_df[col] = farmer_df[col].astype("string").str.strip() 
    mso_df[col] = mso_df[col].astype("string").str.strip()

ตรวจสอบ missing value ใน key ที่จะใช้เชื่อมข้อมูล

In [29]:
farmer_df[area_cols].isna().sum()

province    85
amphur      85
tambon      85
dtype: int64

In [30]:
mso_df[area_cols].isna().sum()

province    0
amphur      0
tambon      0
dtype: int64

ตรวจสอบจำนวนพื้นที่ที่ไม่ซ้ำในแต่ละชุดข้อมูล 

ขั้นตอนนี้ช่วยให้เห็นว่าข้อมูลแต่ละชุดครอบคลุมพื้นที่มากน้อยเพียงใด

In [31]:
farmer_df[area_cols].drop_duplicates().shape

(11, 3)

In [32]:
mso_df[area_cols].drop_duplicates().shape

(99, 3)

## 3.7 สร้าง Column สำหรับใช้วิเคราะห์เพิ่มเติม 
ข้อมูลเกษตรกรมีรายได้และหนี้สินแยกเป็นในภาคเกษตรและนอกภาคเกษตร เช่น 
- `income_in` 
- `income_out` 
- `debts_in` 
- `debts_out` 

เพื่อให้วิเคราะห์ง่ายขึ้น เราจะสร้าง column รวม ได้แก่ 
- `total_income` 
- `total_debt` 

column เหล่านี้จะช่วยให้ query วิเคราะห์รายได้และหนี้สินในระดับพื้นที่ได้ง่ายขึ้น

In [33]:
income_cols = ["income_in", "income_out"] 
debt_cols = ["debts_in", "debts_out"] 

for col in income_cols + debt_cols: 
    farmer_df[col] = pd.to_numeric(farmer_df[col], errors="coerce") 
    
farmer_df["total_income"] = farmer_df[income_cols].sum(axis=1, skipna=True) 
farmer_df["total_debt"] = farmer_df[debt_cols].sum(axis=1, skipna=True)

In [34]:
farmer_df[[ 
    "pid", 
    "province", 
    "amphur", 
    "tambon", 
    "income_in", 
    "income_out", 
    "total_income", 
    "debts_in", 
    "debts_out", 
    "total_debt" 
]].head()

,pid,province,amphur,tambon,income_in,income_out,total_income,debts_in,debts_out,total_debt
0,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,<NA>,<NA>,NaN,NaN,0.0,NaN,NaN,0.0
1,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,<NA>,<NA>,NaN,NaN,0.0,NaN,NaN,0.0
2,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,<NA>,<NA>,NaN,NaN,0.0,NaN,NaN,0.0
3,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,<NA>,<NA>,NaN,NaN,0.0,NaN,NaN,0.0
4,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,<NA>,<NA>,NaN,NaN,0.0,NaN,NaN,0.0


สำหรับข้อมูล MSO เราสามารถใช้ข้อมูลประชากรและข้อมูลกลุ่มเป้าหมายในการวิเคราะห์ เช่น 
- อายุ 
- เพศ 
- ระดับการศึกษา 
- อาชีพหลัก 
- ประเภทกลุ่มเป้าหมาย 
- หน่วยงานของ CM 

ในบทนี้เราจะปรับชนิดข้อมูลของ `age` ให้เป็นตัวเลข เพื่อใช้วิเคราะห์ได้

In [35]:
mso_df["age"] = pd.to_numeric(mso_df["age"])

In [36]:
mso_df[[ 
    "household_id", 
    "house_code", 
    "province", 
    "amphur", 
    "tambon", 
    "age", 
    "gender", 
    "education_level", 
    "main_occupation", 
    "target_group", 
    "cm_department" 
]].head()

,household_id,house_code,province,amphur,tambon,age,gender,education_level,main_occupation,target_group,cm_department
0,648ff7d6b64d06b6b0dc7f02,<NA>,อุดรธานี,นายูง,โนนทอง,3,หญิง,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,62d3f7a6d6f101550054198b,70040223124,ราชบุรี,ดำเนินสะดวก,ท่านัด,55,หญิง,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,62a05c9df5674ecdd3805787,<NA>,หนองคาย,ท่าบ่อ,นาข่า,33,หญิง,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,62908e378fa67bf5ad7d5555,<NA>,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์,5,หญิง,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,6319b26ca3a37508384e36fc,<NA>,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว,3,ชาย,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


## 3.8 สร้าง DuckDB Connection 
DuckDB จะทำหน้าที่เป็น SQL engine สำหรับ query pandas DataFrame 

ข้อดีคือ เราสามารถใช้ SQL กับ DataFrame ได้โดยตรง โดยไม่ต้องนำข้อมูลเข้า database แยกต่างหาก

In [37]:
con = duckdb.connect()

## 3.9 Register DataFrame เป็น Table 
ก่อน query ด้วย SQL ต้อง register DataFrame ให้ DuckDB รู้จักก่อน

ในบทนี้จะใช้ชื่อ table ว่า 
- `farmer` 
- `mso`

In [38]:
con.register("farmer", farmer_df) 
con.register("mso", mso_df)

In [39]:
con.sql(
    """ 
    SELECT * 
    FROM farmer 
    LIMIT 5 
    """
).df()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet,total_income,total_debt
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,0.0,0.0
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,0.0,0.0
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,0.0,0.0
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,0.0,0.0
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,0.0,0.0


In [40]:
con.sql(
    """ 
    SELECT * 
    FROM mso 
    LIMIT 5 
    """
).df()

,last_updated_at,household_id,house_code,household_created_at,age,gender,tambon,amphur,province,education_level,main_occupation,target_group,cm_id,cm_department
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,NaN,2023-06-19 13:38:14.881,3,หญิง,โนนทอง,นายูง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ท่านัด,ดำเนินสะดวก,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,NaN,2022-08-06 15:23:57.850,33,หญิง,นาข่า,ท่าบ่อ,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,NaN,2022-05-27 15:39:19.197,5,หญิง,โดมประดิษฐ์,น้ำยืน,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,NaN,2022-08-09 16:14:20.652,3,ชาย,หนองหญ้าขาว,สีคิ้ว,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


## 3.10 SQL พื้นฐาน: SELECT 
`SELECT` ใช้เลือก column ที่ต้องการดูจากตาราง 

ตัวอย่าง: เลือกข้อมูลพื้นที่และหน่วยงานจากข้อมูลเกษตรกร

In [41]:
con.sql(
    """ 
    SELECT 
        department, 
        source_sheet, 
        province, 
        amphur, 
        tambon, 
        farmer_type, 
        main_occupation 
    FROM farmer 
    LIMIT 10 
    """
).df()

,department,source_sheet,province,amphur,tambon,farmer_type,main_occupation
0,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
1,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
2,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
3,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
4,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
5,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
6,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
7,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
8,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
9,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None


ตัวอย่าง: เลือกข้อมูลพื้นที่และกลุ่มเป้าหมายจากข้อมูล MSO

In [42]:
con.sql(
    """ 
    SELECT 
        province, 
        amphur, 
        tambon, 
        age, 
        gender, 
        education_level, 
        main_occupation, 
        target_group, 
        cm_department 
    FROM mso 
    LIMIT 10 
    """
).df()

,province,amphur,tambon,age,gender,education_level,main_occupation,target_group,cm_department
0,อุดรธานี,นายูง,โนนทอง,3,หญิง,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,ราชบุรี,ดำเนินสะดวก,ท่านัด,55,หญิง,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,หนองคาย,ท่าบ่อ,นาข่า,33,หญิง,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์,5,หญิง,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว,3,ชาย,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา
5,ประจวบคีรีขันธ์,เมืองประจวบคีรีขันธ์,อ่าวน้อย,39,ชาย,มัธยมศึกษาตอนต้น,ธุรกิจส่วนตัว/ค้าขาย/เจ้าของกิจการ,วัยผู้ใหญ่/วัยแรงงาน,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
6,อุดรธานี,เทศบาลนครอุดรธานี,หมากแข้ง,4,หญิง,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
7,นครศรีธรรมราช,ปากพนัง,ปากพนังฝั่งตะวันออก,68,NaN,NaN,ธุรกิจส่วนตัว/ค้าขาย/เจ้าของกิจการ,ผู้สูงอายุ,ศูนย์คุ้มครองคนไร้ที่พึ่ง จ.นครศรีธรรมราช
8,สระแก้ว,วัฒนานคร,วัฒนานคร,90,หญิง,ประถมศึกษา,NaN,ผู้สูงอายุ,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
9,บุรีรัมย์,หนองหงส์,ไทยสามัคคี,6,หญิง,ไม่ได้เรียนหนังสือ,NaN,เด็ก,บ้านพักเด็กและครอบครัวจังหวัดบุรีรัมย์


## 3.11 SQL พื้นฐาน: WHERE 
`WHERE` ใช้กรองข้อมูลตามเงื่อนไข 

ตัวอย่าง: เลือกข้อมูลเกษตรกรเฉพาะจังหวัดอุดรธานี

In [43]:
con.sql(
    """ 
    SELECT 
        department, 
        province, 
        amphur, 
        tambon, 
        farmer_type, 
        total_income, 
        total_debt 
    FROM farmer 
    WHERE province = 'อุดรธานี' 
    LIMIT 10 """
).df()

,department,province,amphur,tambon,farmer_type,total_income,total_debt
0,กรมปศุสัตว์,อุดรธานี,เพ็ญ,นาพู่,เกษตรกรผู้เลี้ยงสัตว์,0.0,0.0


## 3.12 SQL พื้นฐาน: ORDER BY 
`ORDER BY` ใช้เรียงข้อมูล 

ตัวอย่าง: เรียงพื้นที่ตามรายได้รวมจากน้อยไปมาก

In [44]:
con.sql(
    """ 
    SELECT 
        province, 
        amphur, 
        tambon, 
        total_income, 
        total_debt 
    FROM farmer 
    WHERE province IS NOT NULL 
    ORDER BY total_income ASC LIMIT 10 
    """
).df()

,province,amphur,tambon,total_income,total_debt
0,บุรีรัมย์,นางรอง,ก้านเหลือง,0.0,0.0
1,อุบลราชธานี,ตระการพืชผล,หนองเต่า,0.0,0.0
2,อุดรธานี,เพ็ญ,นาพู่,0.0,0.0
3,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,0.0,0.0
4,แพร่,ร้องกวาง,แม่ยางร้อง,0.0,0.0
5,อุบลราชธานี,ตระการพืชผล,สะพือ,0.0,0.0
6,ขอนแก่น,บ้านไผ่,ป่าปอ,0.0,0.0
7,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,0.0,0.0
8,สุรินทร์,รัตนบุรี,เบิด,0.0,0.0
9,พะเยา,ภูซาง,ทุ่งกล้วย,0.0,0.0


หากต้องการเรียงจากมากไปน้อย ใช้ `DESC` 

ตัวอย่าง: เรียงพื้นที่ตามหนี้สินรวมจากมากไปน้อย

In [45]:
con.sql(
    """ 
    SELECT 
        province, 
        amphur, 
        tambon, 
        total_income, 
        total_debt 
    FROM farmer 
    WHERE province IS NOT NULL 
    ORDER BY total_debt 
    DESC LIMIT 10 
    """
).df()

,province,amphur,tambon,total_income,total_debt
0,บุรีรัมย์,นางรอง,ก้านเหลือง,0.0,0.0
1,อุบลราชธานี,ตระการพืชผล,หนองเต่า,0.0,0.0
2,อุดรธานี,เพ็ญ,นาพู่,0.0,0.0
3,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,0.0,0.0
4,แพร่,ร้องกวาง,แม่ยางร้อง,0.0,0.0
5,อุบลราชธานี,ตระการพืชผล,สะพือ,0.0,0.0
6,ขอนแก่น,บ้านไผ่,ป่าปอ,0.0,0.0
7,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,0.0,0.0
8,สุรินทร์,รัตนบุรี,เบิด,0.0,0.0
9,พะเยา,ภูซาง,ทุ่งกล้วย,0.0,0.0


## 3.13 SQL พื้นฐาน: GROUP BY 
`GROUP BY` ใช้สรุปข้อมูลตามกลุ่ม 

ตัวอย่าง: สรุปจำนวน record เกษตรกรตามจังหวัดและหน่วยงาน

In [46]:
con.sql(
    """ 
    SELECT 
        province, 
        department, 
        COUNT(*) AS farmer_records, 
        COUNT(DISTINCT pid) AS unique_farmers 
    FROM farmer 
    GROUP BY province, department 
    ORDER BY farmer_records DESC 
    LIMIT 10 
    """
).df()

,province,department,farmer_records,unique_farmers
0,NaN,กรมส่งเสริมการเกษตร,70,70
1,NaN,กรมส่งเสริมสหกรณ์,15,15
2,อุบลราชธานี,กรมปศุสัตว์,3,3
3,อุดรธานี,กรมปศุสัตว์,1,1
4,สุรินทร์,กรมปศุสัตว์,1,1
5,ขอนแก่น,กรมปศุสัตว์,1,1
6,พะเยา,กรมปศุสัตว์,1,1
7,ศรีสะเกษ,กรมปศุสัตว์,1,1
8,แพร่,กรมปศุสัตว์,1,1
9,บุรีรัมย์,กรมปศุสัตว์,1,1


ตัวอย่าง: สรุปจำนวน record MSO ตามจังหวัดและกลุ่มเป้าหมาย

In [47]:
con.sql(
    """ 
    SELECT 
        province, 
        target_group, 
        COUNT(*) AS mso_records, 
        COUNT(DISTINCT household_id) AS unique_households 
    FROM mso 
    GROUP BY province, target_group 
    ORDER BY mso_records 
    DESC LIMIT 10 
    """
).df()

,province,target_group,mso_records,unique_households
0,หนองคาย,วัยผู้ใหญ่/วัยแรงงาน,4,4
1,เชียงใหม่,วัยผู้ใหญ่/วัยแรงงาน,3,3
2,ขอนแก่น,ผู้สูงอายุ,3,3
3,นครศรีธรรมราช,วัยผู้ใหญ่/วัยแรงงาน,3,3
4,อุทัยธานี,วัยผู้ใหญ่/วัยแรงงาน,3,3
5,นครศรีธรรมราช,ผู้สูงอายุ,3,3
6,อุดรธานี,เด็กเล็ก,3,3
7,อุดรธานี,วัยผู้ใหญ่/วัยแรงงาน,3,3
8,นครราชสีมา,ผู้สูงอายุ,3,3
9,ราชบุรี,วัยผู้ใหญ่/วัยแรงงาน,2,2


## 3.14 Aggregate Function ที่ใช้บ่อย 
Aggregate function คือ function สำหรับสรุปข้อมูล 

| Function | ความหมาย | ตัวอย่างการใช้ | 
|---|---|---| 
| `COUNT(*)` | นับจำนวนแถว | จำนวน record ทั้งหมด | 
| `COUNT(DISTINCT col)` | นับจำนวนค่าที่ไม่ซ้ำ | จำนวนบุคคลหรือครัวเรือนไม่ซ้ำ | 
| `SUM(col)` | ผลรวม | ผลรวมรายได้หรือหนี้สิน | 
| `AVG(col)` | ค่าเฉลี่ย | รายได้เฉลี่ยหรืออายุเฉลี่ย | 
| `MIN(col)` | ค่าน้อยที่สุด | อายุน้อยที่สุด | 
| `MAX(col)` | ค่ามากที่สุด | อายุมากที่สุด | ในการวิเคราะห์ข้อมูลที่มีรหัสบุคคลหรือรหัสครัวเรือน 

ควรระวังความแตกต่างระหว่างจำนวนแถวกับจำนวนค่าที่ไม่ซ้ำ

In [48]:
con.sql(
    """ 
    SELECT 
        province, 
        COUNT(*) AS farmer_records, 
        COUNT(DISTINCT pid) AS unique_farmers, 
        AVG(total_income) AS avg_total_income, 
        AVG(total_debt) AS avg_total_debt, 
        SUM(total_income) AS sum_total_income, 
        SUM(total_debt) AS sum_total_debt 
    FROM farmer 
    GROUP BY province 
    ORDER BY farmer_records DESC 
    LIMIT 10 
    """
).df()

,province,farmer_records,unique_farmers,avg_total_income,avg_total_debt,sum_total_income,sum_total_debt
0,NaN,85,85,66383.529412,29941.176471,5642600.0,2545000.0
1,อุบลราชธานี,3,3,0.000000,0.000000,0.0,0.0
2,อุดรธานี,1,1,0.000000,0.000000,0.0,0.0
3,ศรีสะเกษ,1,1,0.000000,0.000000,0.0,0.0
4,ขอนแก่น,1,1,0.000000,0.000000,0.0,0.0
5,บุรีรัมย์,1,1,0.000000,0.000000,0.0,0.0
6,แพร่,1,1,0.000000,0.000000,0.0,0.0
7,พะเยา,1,1,0.000000,0.000000,0.0,0.0
8,สุรินทร์,1,1,0.000000,0.000000,0.0,0.0


In [49]:
con.sql(
    """ 
    SELECT 
        province, 
        COUNT(*) AS mso_records, 
        COUNT(DISTINCT household_id) AS unique_households, 
        AVG(age) AS avg_age, 
        MIN(age) AS min_age, 
        MAX(age) AS max_age 
    FROM mso 
    GROUP BY province 
    ORDER BY mso_records DESC 
    LIMIT 10 
    """
).df()

,province,mso_records,unique_households,avg_age,min_age,max_age
0,นครศรีธรรมราช,8,8,39.625000,5,68
1,อุดรธานี,7,7,27.142857,3,62
2,เชียงใหม่,5,5,30.600000,3,54
3,หนองคาย,4,4,36.500000,31,44
4,นครราชสีมา,4,4,61.000000,3,103
5,บุรีรัมย์,4,4,30.000000,6,79
6,อุทัยธานี,4,4,53.000000,38,65
7,ขอนแก่น,4,4,70.500000,58,83
8,นครนายก,4,4,36.750000,13,70
9,นครพนม,3,3,38.666667,17,54


## 3.15 ทำไมต้อง JOIN ข้อมูล 
ข้อมูลแต่ละชุดตอบคำถามได้เพียงบางส่วน 

ข้อมูล `farmer` อาจตอบคำถามได้ เช่น 
- จังหวัดใดมีเกษตรกรเปราะบางจำนวนมาก 
- หน่วยงานใดมีข้อมูลเกษตรกรจำนวนมาก 
- พื้นที่ใดมีรายได้เฉลี่ยต่ำหรือหนี้สินเฉลี่ยสูง 
- เกษตรกรแต่ละประเภทกระจายอยู่ในพื้นที่ใด 

ข้อมูล `mso` อาจตอบคำถามได้ เช่น 
- จังหวัดใดมีครัวเรือนใน logbook จำนวนมาก 
- กลุ่มเป้าหมายใดพบมากในแต่ละพื้นที่ 
- พื้นที่ใดมีข้อมูลจาก CM หรือหน่วยงานสังคมมาก

แต่ถ้าต้องการตอบคำถามร่วม เช่น 

> พื้นที่ใดมีทั้งข้อมูลเกษตรกรเปราะบางและข้อมูลครัวเรือนจาก MSO 

หรือ 

> พื้นที่ใดมีจำนวนเกษตรกรสูง แต่ข้อมูล MSO ต่ำ เราต้องเชื่อมข้อมูลทั้งสองชุดเข้าด้วยกัน

## 3.16 ประเภทของ JOIN และความหมายของผลลัพธ์ 
การ JOIN คือการเชื่อมข้อมูลจาก 2 ตารางขึ้นไป โดยใช้ key ที่มีความหมายร่วมกัน 

ในบทนี้ key ที่ใช้เชื่อมคือข้อมูลพื้นที่ ได้แก่ 
- `province` 
- `amphur` 
- `tambon` 

ชนิดของ JOIN ที่พบบ่อย ได้แก่ 
| JOIN Type | ความหมาย | เหมาะกับคำถามแบบใด | 
|---|---|---| 
| `INNER JOIN` | เอาเฉพาะข้อมูลที่พบตรงกันทั้งสองฝั่ง | ต้องการดูเฉพาะพื้นที่ที่มีข้อมูลทั้ง farmer และ mso | 
| `LEFT JOIN` | เก็บข้อมูลฝั่งซ้ายไว้ทั้งหมด และเติมข้อมูลฝั่งขวาถ้ามี | ต้องการรักษาพื้นที่จาก farmer ไว้ทั้งหมด | 
| `RIGHT JOIN` | เก็บข้อมูลฝั่งขวาไว้ทั้งหมด และเติมข้อมูลฝั่งซ้ายถ้ามี | ต้องการรักษาพื้นที่จาก mso ไว้ทั้งหมด | 
| `FULL OUTER JOIN` | เก็บข้อมูลจากทั้งสองฝั่ง ไม่ว่าจะ match กันหรือไม่ | ต้องการดู coverage ของข้อมูลทั้งสองแหล่ง | 

การเลือก JOIN ต้องเริ่มจากคำถามวิเคราะห์ ไม่ใช่เริ่มจาก syntax

## 3.17 สร้าง Summary 
ก่อน JOIN ก่อน JOIN ข้อมูลจากหลายแหล่ง ควรสรุปข้อมูลให้อยู่ในระดับเดียวกันก่อน 

ในตัวอย่างนี้ เราจะสรุปทั้งสองชุดข้อมูลให้อยู่ในระดับ 

`province + amphur + tambon` 

เหตุผลคือข้อมูลทั้งสองชุดมี grain ต่างกัน 
- `farmer` อาจมี 1 แถวต่อบุคคลหรือรายการเกษตรกร 
- `mso` อาจมี 1 แถวต่อสมาชิกครัวเรือนหรือรายการ logbook 

ถ้า JOIN แบบ row-level ทันที ข้อมูลอาจพองหรือซ้ำโดยไม่ตั้งใจ 

ดังนั้นแนวทางที่ปลอดภัยสำหรับบทนี้คือ 

1. สรุปข้อมูล farmer ตามพื้นที่ 
2. สรุปข้อมูล mso ตามพื้นที่ 
3. JOIN summary ทั้งสองชุดเข้าด้วยกัน

In [50]:
con.sql(
    """ 
    SELECT 
        province, 
        amphur, 
        tambon, 
        COUNT(*) AS farmer_records, 
        COUNT(DISTINCT pid) AS unique_farmers, 
        AVG(total_income) AS avg_total_income, 
        AVG(total_debt) AS avg_total_debt 
    FROM farmer 
    GROUP BY 
        province, 
        amphur, 
        tambon 
    ORDER BY farmer_records DESC 
    LIMIT 10 
    """
).df()

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt
0,NaN,NaN,NaN,85,85,66383.529412,29941.176471
1,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1,0.000000,0.000000
2,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1,0.000000,0.000000
3,อุดรธานี,เพ็ญ,นาพู่,1,1,0.000000,0.000000
4,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,1,0.000000,0.000000
5,แพร่,ร้องกวาง,แม่ยางร้อง,1,1,0.000000,0.000000
6,อุบลราชธานี,ตระการพืชผล,สะพือ,1,1,0.000000,0.000000
7,ขอนแก่น,บ้านไผ่,ป่าปอ,1,1,0.000000,0.000000
8,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,1,0.000000,0.000000
9,พะเยา,ภูซาง,ทุ่งกล้วย,1,1,0.000000,0.000000


In [51]:
con.sql(
    """ 
    SELECT 
        province, 
        amphur, 
        tambon, 
        COUNT(*) AS mso_records, 
        COUNT(DISTINCT household_id) AS unique_households, 
        AVG(age) AS avg_age 
    FROM mso 
    GROUP BY 
        province, 
        amphur, 
        tambon 
    ORDER BY mso_records DESC 
    LIMIT 10 
    """
).df()

,province,amphur,tambon,mso_records,unique_households,avg_age
0,ตาก,แม่สอด,แม่กาษา,2,2,32.0
1,หนองคาย,ท่าบ่อ,นาข่า,1,1,33.0
2,นครศรีธรรมราช,ปากพนัง,ปากพนังฝั่งตะวันออก,1,1,68.0
3,ราชบุรี,ดำเนินสะดวก,ท่านัด,1,1,55.0
4,สุพรรณบุรี,บางปลาม้า,มะขามล้ม,1,1,4.0
5,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์,1,1,5.0
6,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว,1,1,3.0
7,ประจวบคีรีขันธ์,เมืองประจวบคีรีขันธ์,อ่าวน้อย,1,1,39.0
8,บุรีรัมย์,หนองหงส์,ไทยสามัคคี,1,1,6.0
9,เชียงใหม่,เทศบาลตำบลเมืองนะ,เมืองนะ,1,1,3.0


## 3.18 INNER JOIN: ดูเฉพาะพื้นที่ที่มีข้อมูลทั้งสองฝั่ง

`INNER JOIN` จะคืนเฉพาะพื้นที่ที่พบทั้งในข้อมูล farmer และ mso

เหมาะกับคำถาม เช่น

> พื้นที่ใดมีทั้งข้อมูลเกษตรกรเปราะบางและข้อมูลครัวเรือนจาก MSO

ข้อควรระวังคือ พื้นที่ที่มีเฉพาะข้อมูล farmer หรือเฉพาะข้อมูล mso จะไม่ปรากฏในผลลัพธ์

In [52]:
con.sql(
    """ 
    WITH farmer_area AS ( 
        SELECT 
            province, 
            amphur, 
            tambon, 
            COUNT(*) AS farmer_records, 
            COUNT(DISTINCT pid) AS unique_farmers, 
            AVG(total_income) AS avg_total_income, 
            AVG(total_debt) AS avg_total_debt 
        FROM farmer 
        GROUP BY 
            province, 
            amphur, 
            tambon 
    ),

    mso_area AS ( 
        SELECT 
            province, 
            amphur, 
            tambon, 
            COUNT(*) AS mso_records, 
            COUNT(DISTINCT household_id) AS unique_households, 
            AVG(age) AS avg_age 
        FROM mso 
        GROUP BY 
            province,
            amphur, 
            tambon 
    )

    SELECT 
        f.province, 
        f.amphur, 
        f.tambon, 
        f.farmer_records, 
        f.unique_farmers, 
        f.avg_total_income, 
        f.avg_total_debt, 
        m.mso_records, 
        m.unique_households, 
        m.avg_age 
    FROM farmer_area f 
    INNER JOIN mso_area m 
        ON f.province = m.province 
        AND f.amphur = m.amphur 
        AND f.tambon = m.tambon 
    ORDER BY f.farmer_records DESC 
    LIMIT 20 
    """
).df()

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt,mso_records,unique_households,avg_age


## 3.19 LEFT JOIN: รักษาข้อมูล farmer ไว้ทั้งหมด 
`LEFT JOIN` จะเก็บข้อมูลฝั่งซ้ายไว้ทั้งหมด 

ในตัวอย่างนี้ ถ้าใช้ `farmer_area` เป็นฝั่งซ้าย ผลลัพธ์จะมีพื้นที่ทั้งหมดที่พบในข้อมูล farmer 

ถ้าพื้นที่ใดไม่มีข้อมูลจาก mso ค่า column ฝั่ง mso จะเป็น `NULL` เหมาะกับคำถาม เช่น 

> พื้นที่ใดมีข้อมูลเกษตรกร แต่ยังไม่พบข้อมูลจาก MSO 

หรือ 

> หากเริ่มจากฐานข้อมูลเกษตรกร เราสามารถเติมข้อมูล MSO เข้ามาได้มากน้อยแค่ไหน

In [53]:
con.sql(
    """ 
    WITH farmer_area AS ( 
        SELECT 
            province, 
            amphur, 
            tambon, 
            COUNT(*) AS farmer_records, 
            COUNT(DISTINCT pid) AS unique_farmers, 
            AVG(total_income) AS avg_total_income, 
            AVG(total_debt) AS avg_total_debt 
        FROM farmer 
        GROUP BY 
            province, 
            amphur, 
            tambon 
    ), 
    
    mso_area AS ( 
        SELECT 
            province, 
            amphur, 
            tambon, 
            COUNT(*) AS mso_records, 
            COUNT(DISTINCT household_id) AS unique_households, 
            AVG(age) AS avg_age 
        FROM mso 
        GROUP BY 
            province, 
            amphur, 
            tambon 
    ) 
    
    SELECT 
        f.province, 
        f.amphur, 
        f.tambon, 
        f.farmer_records, 
        f.unique_farmers, 
        f.avg_total_income, 
        f.avg_total_debt, 
        m.mso_records, 
        m.unique_households, 
        m.avg_age 
    FROM farmer_area f 
    LEFT JOIN mso_area m 
        ON f.province = m.province 
        AND f.amphur = m.amphur 
        AND f.tambon = m.tambon 
    ORDER BY f.farmer_records DESC 
    LIMIT 20 
    """
).df()

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt,mso_records,unique_households,avg_age
0,NaN,NaN,NaN,85,85,66383.529412,29941.176471,<NA>,<NA>,NaN
1,สุรินทร์,รัตนบุรี,เบิด,1,1,0.000000,0.000000,<NA>,<NA>,NaN
2,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1,0.000000,0.000000,<NA>,<NA>,NaN
3,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1,0.000000,0.000000,<NA>,<NA>,NaN
4,แพร่,ร้องกวาง,แม่ยางร้อง,1,1,0.000000,0.000000,<NA>,<NA>,NaN
5,อุบลราชธานี,ตระการพืชผล,สะพือ,1,1,0.000000,0.000000,<NA>,<NA>,NaN
6,อุดรธานี,เพ็ญ,นาพู่,1,1,0.000000,0.000000,<NA>,<NA>,NaN
7,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,1,0.000000,0.000000,<NA>,<NA>,NaN
8,ขอนแก่น,บ้านไผ่,ป่าปอ,1,1,0.000000,0.000000,<NA>,<NA>,NaN
9,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,1,0.000000,0.000000,<NA>,<NA>,NaN


## 3.20 FULL OUTER JOIN: ตรวจสอบ Coverage ของข้อมูลทั้งสองแหล่ง 
`FULL OUTER JOIN` จะเก็บข้อมูลจากทั้งสองฝั่งทั้งหมด 

เหมาะกับคำถาม เช่น 

> พื้นที่ใดมีข้อมูลทั้งสองฝั่ง พื้นที่ใดมีเฉพาะ farmer และพื้นที่ใดมีเฉพาะ mso 

ผลลัพธ์ลักษณะนี้เหมาะสำหรับตรวจสอบ coverage ของข้อมูลก่อนนำไปวิเคราะห์ลึกขึ้น

In [54]:
con.sql(
    """ 
    WITH farmer_area AS ( 
        SELECT 
            province, 
            amphur, 
            tambon, 
            COUNT(*) AS farmer_records, 
            COUNT(DISTINCT pid) AS unique_farmers 
        FROM farmer 
        GROUP BY 
            province, 
            amphur, 
            tambon 
    ), 
    
    mso_area AS ( 
        SELECT 
            province, 
            amphur, 
            tambon, 
            COUNT(*) AS mso_records, 
            COUNT(DISTINCT household_id) AS unique_households 
        FROM mso 
        GROUP BY 
            province, 
            amphur, 
            tambon 
        ) 
        
        SELECT 
            COALESCE(f.province, m.province) AS province, 
            COALESCE(f.amphur, m.amphur) AS amphur, 
            COALESCE(f.tambon, m.tambon) AS tambon, 
            f.farmer_records, 
            f.unique_farmers, 
            m.mso_records, 
            m.unique_households, 
            CASE 
                WHEN f.province IS NOT NULL AND m.province IS NOT NULL THEN 'matched_both' 
                WHEN f.province IS NOT NULL AND m.province IS NULL THEN 'farmer_only' 
                WHEN f.province IS NULL AND m.province IS NOT NULL THEN 'mso_only' 
            END AS match_status 
        FROM farmer_area f 
        FULL OUTER JOIN mso_area m 
            ON f.province = m.province 
            AND f.amphur = m.amphur 
            AND f.tambon = m.tambon 
        ORDER BY 
            match_status, 
            province, 
            amphur, 
            tambon 
        LIMIT 30 
        """
    ).df()

,province,amphur,tambon,farmer_records,unique_farmers,mso_records,unique_households,match_status
0,ขอนแก่น,บ้านไผ่,ป่าปอ,1,1,<NA>,<NA>,farmer_only
1,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1,<NA>,<NA>,farmer_only
2,พะเยา,ภูซาง,ทุ่งกล้วย,1,1,<NA>,<NA>,farmer_only
3,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,1,<NA>,<NA>,farmer_only
4,สุรินทร์,รัตนบุรี,เบิด,1,1,<NA>,<NA>,farmer_only
5,อุดรธานี,เพ็ญ,นาพู่,1,1,<NA>,<NA>,farmer_only
6,อุบลราชธานี,ตระการพืชผล,สะพือ,1,1,<NA>,<NA>,farmer_only
7,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1,<NA>,<NA>,farmer_only
8,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,1,<NA>,<NA>,farmer_only
9,แพร่,ร้องกวาง,แม่ยางร้อง,1,1,<NA>,<NA>,farmer_only


## 3.21 ใช้ COALESCE เพื่อจัดการค่า NULL หลัง JOIN 
หลัง JOIN ถ้าพื้นที่ฝั่ง farmer ไม่มีข้อมูลตรงกับฝั่ง mso ค่า metric จาก mso จะเป็น `NULL` 

ในการวิเคราะห์เชิงจำนวน เรามักต้องการเปลี่ยน `NULL` เป็น 0 เพื่อให้คำนวณต่อได้ 

ตัวอย่าง: 
- ถ้า `mso_records` เป็น `NULL` หลัง LEFT JOIN อาจตีความใน summary ได้ว่าไม่พบ record จาก mso ในข้อมูลที่มีอยู่ 
- แต่ต้องระวังว่า 0 ในที่นี้หมายถึง “ไม่พบในข้อมูล” ไม่ใช่การยืนยันว่าในความเป็นจริงไม่มีเคส

In [55]:
con.sql(
    """ 
    WITH farmer_area AS ( 
        SELECT 
            province, 
            amphur, 
            tambon, 
            COUNT(*) AS farmer_records, 
            COUNT(DISTINCT pid) AS unique_farmers, 
            AVG(total_income) AS avg_total_income, 
            AVG(total_debt) AS avg_total_debt 
        FROM farmer 
        GROUP BY 
            province, 
            amphur, 
            tambon 
    ), 
    
    mso_area AS ( 
        SELECT 
            province, 
            amphur, 
            tambon, 
            COUNT(*) AS mso_records, 
            COUNT(DISTINCT household_id) AS unique_households, 
            AVG(age) AS avg_age 
        FROM mso 
        GROUP BY 
            province, 
            amphur, 
            tambon 
    ), 
    
    joined_area AS ( 
        SELECT 
            f.province, 
            f.amphur, 
            f.tambon, 
            f.farmer_records, 
            f.unique_farmers, 
            f.avg_total_income, 
            f.avg_total_debt, 
            COALESCE(m.mso_records, 0) AS mso_records, 
            COALESCE(m.unique_households, 0) AS unique_households, 
            m.avg_age 
        FROM farmer_area f 
        LEFT JOIN mso_area m 
            ON f.province = m.province 
            AND f.amphur = m.amphur 
            AND f.tambon = m.tambon 
    ) 
    
    SELECT 
        *, 
        farmer_records + mso_records AS total_related_records, 
        farmer_records - mso_records AS gap_score, 
        CASE 
            WHEN mso_records > 0 THEN 'has_mso_data' 
            ELSE 'farmer_only' 
        END AS mso_match_status 
    FROM joined_area 
    ORDER BY total_related_records DESC 
    LIMIT 20 
    """
).df()

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt,mso_records,unique_households,avg_age,total_related_records,gap_score,mso_match_status
0,NaN,NaN,NaN,85,85,66383.529412,29941.176471,0,0,NaN,85,85,farmer_only
1,สุรินทร์,รัตนบุรี,เบิด,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
2,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
3,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
4,แพร่,ร้องกวาง,แม่ยางร้อง,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
5,อุบลราชธานี,ตระการพืชผล,สะพือ,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
6,อุดรธานี,เพ็ญ,นาพู่,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
7,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
8,ขอนแก่น,บ้านไผ่,ป่าปอ,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
9,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only


## 3.22 ตรวจสอบผลลัพธ์หลัง JOIN 
หลัง JOIN ควรตรวจสอบว่า join แล้วเกิดอะไรขึ้นกับข้อมูล 

ตัวอย่างคำถามที่ควรตรวจ: 
1. พื้นที่จาก farmer มีทั้งหมดกี่พื้นที่ 
2. พื้นที่ที่ match กับ mso ได้มีกี่พื้นที่ 
3. พื้นที่ที่มีเฉพาะ farmer มีกี่พื้นที่ 
4. พื้นที่ที่มีเฉพาะ mso มีกี่พื้นที่ 
5. มีค่า NULL เกิดขึ้นใน column ใดบ้าง 

การตรวจสอบนี้ช่วยให้ผู้วิเคราะห์ไม่สรุปผลผิดจากปัญหา key ไม่ตรงกัน

In [56]:
con.sql(
    """ 
    WITH farmer_area AS ( 
        SELECT 
            province, 
            amphur, 
            tambon, 
            COUNT(*) AS farmer_records 
        FROM farmer 
        GROUP BY 
            province, 
            amphur, 
            tambon 
    ), 
    
    mso_area AS ( 
        SELECT 
            province, 
            amphur, 
            tambon, 
            COUNT(*) AS mso_records 
        FROM mso 
        GROUP BY 
            province, 
            amphur, 
            tambon 
    ), 
    
    coverage AS ( 
        SELECT 
            COALESCE(f.province, m.province) AS province, 
            COALESCE(f.amphur, m.amphur) AS amphur, 
            COALESCE(f.tambon, m.tambon) AS tambon, 
            f.farmer_records, m.mso_records, 
            CASE 
                WHEN f.province IS NOT NULL AND m.province IS NOT NULL THEN 'matched_both' 
                WHEN f.province IS NOT NULL AND m.province IS NULL THEN 'farmer_only' 
                WHEN f.province IS NULL AND m.province IS NOT NULL THEN 'mso_only' 
            END AS match_status 
        FROM farmer_area f 
        FULL OUTER JOIN mso_area m 
            ON f.province = m.province 
            AND f.amphur = m.amphur 
            AND f.tambon = m.tambon 
    ) 
            
    SELECT 
        match_status, 
        COUNT(*) AS area_count 
    FROM coverage 
    GROUP BY match_status 
    ORDER BY area_count DESC 
    """
).df()

,match_status,area_count
0,mso_only,99
1,farmer_only,10
2,NaN,1


## 3.24 เก็บ Query เป็น Python Function 
เมื่อ query เริ่มยาว และต้องใช้งานซ้ำ ควรเก็บ query เป็น Python function 

ข้อดีคือ 
- เรียกใช้งานซ้ำได้ 
- ลดการ copy query 
- ใช้ parameter เพื่อปรับเงื่อนไขได้ 
- เหมาะกับการนำไปใช้ใน workflow การวิเคราะห์จริง

In [57]:
def build_area_join_summary(con): 
    query = """ 
        WITH farmer_area AS ( 
            SELECT 
                province, 
                amphur, 
                tambon, 
                COUNT(*) AS farmer_records, 
                COUNT(DISTINCT pid) AS unique_farmers, 
                AVG(total_income) AS avg_total_income, 
                AVG(total_debt) AS avg_total_debt 
            FROM farmer 
            GROUP BY 
                province, 
                amphur, 
                tambon 
        ), 
        
        mso_area AS ( 
            SELECT 
                province, 
                amphur, 
                tambon, 
                COUNT(*) AS mso_records, 
                COUNT(DISTINCT household_id) AS unique_households, 
                AVG(age) AS avg_age 
            FROM mso 
            GROUP BY 
                province, 
                amphur, 
                tambon 
        ), 
        
        joined_area AS ( 
            SELECT 
            f.province, 
            f.amphur, 
            f.tambon, 
            f.farmer_records, 
            f.unique_farmers, 
            f.avg_total_income, 
            f.avg_total_debt, 
            COALESCE(m.mso_records, 0) AS mso_records, 
            COALESCE(m.unique_households, 0) AS unique_households, 
            m.avg_age 
        FROM farmer_area f 
        LEFT JOIN mso_area m 
            ON f.province = m.province 
            AND f.amphur = m.amphur 
            AND f.tambon = m.tambon 
        ) 
        
        SELECT 
            *, 
            farmer_records + mso_records AS total_related_records, 
            farmer_records - mso_records AS gap_score, 
            CASE 
                WHEN mso_records > 0 THEN 'has_mso_data' 
                ELSE 'farmer_only' 
            END AS mso_match_status 
        FROM joined_area 
        ORDER BY total_related_records DESC 
        """ 
    return con.sql(query).df()

In [58]:
area_join_summary_df = build_area_join_summary(con) 
area_join_summary_df.head(10)

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt,mso_records,unique_households,avg_age,total_related_records,gap_score,mso_match_status
0,NaN,NaN,NaN,85,85,66383.529412,29941.176471,0,0,NaN,85,85,farmer_only
1,สุรินทร์,รัตนบุรี,เบิด,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
2,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
3,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
4,อุดรธานี,เพ็ญ,นาพู่,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
5,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
6,แพร่,ร้องกวาง,แม่ยางร้อง,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
7,อุบลราชธานี,ตระการพืชผล,สะพือ,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
8,ขอนแก่น,บ้านไผ่,ป่าปอ,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only
9,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,1,0.000000,0.000000,0,0,NaN,1,1,farmer_only


## 3.25 Function แบบรับ Parameter 
ในงานจริง ผู้ใช้มักต้องการวิเคราะห์เฉพาะบางจังหวัด 

เราจึงสามารถสร้าง function ที่รับชื่อจังหวัดเป็น parameter ได้ 

ในตัวอย่างนี้ใช้ parameter binding ผ่าน `?` เพื่อหลีกเลี่ยงการต่อ string เข้า SQL โดยตรง

In [59]:
def build_area_join_summary_by_province(con, province_name): 
    query = """ 
        WITH farmer_area AS ( 
            SELECT 
                province, 
                amphur, 
                tambon, 
                COUNT(*) AS farmer_records, 
                COUNT(DISTINCT pid) AS unique_farmers, 
                AVG(total_income) AS avg_total_income, 
                AVG(total_debt) AS avg_total_debt 
            FROM farmer 
            WHERE province = ? 
            GROUP BY 
                province, 
                amphur, 
                tambon 
        ), 
        
        mso_area AS ( 
            SELECT 
                province, 
                amphur, 
                tambon, 
                COUNT(*) AS mso_records, 
                COUNT(DISTINCT household_id) AS unique_households, 
                AVG(age) AS avg_age 
            FROM mso 
            WHERE province = ? 
            GROUP BY 
                province, 
                amphur, 
                tambon 
        ), 
        
        joined_area AS ( 
            SELECT 
                f.province, 
                f.amphur, 
                f.tambon, 
                f.farmer_records, 
                f.unique_farmers, 
                f.avg_total_income, 
                f.avg_total_debt, 
                COALESCE(m.mso_records, 0) AS mso_records, 
                COALESCE(m.unique_households, 0) AS unique_households, 
                m.avg_age 
            FROM farmer_area f 
            LEFT JOIN mso_area m 
                ON f.province = m.province 
                AND f.amphur = m.amphur 
                AND f.tambon = m.tambon 
        ) 
        
        SELECT 
            *, 
            farmer_records + mso_records AS total_related_records, 
            farmer_records - mso_records AS gap_score, 
            CASE 
                WHEN mso_records > 0 THEN 'has_mso_data' 
                ELSE 'farmer_only' 
            END AS mso_match_status 
        FROM joined_area 
        ORDER BY total_related_records DESC 
        """ 
    return con.execute(query, [province_name, province_name]).df()

In [60]:
build_area_join_summary_by_province(con, "อุดรธานี").head(10)

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt,mso_records,unique_households,avg_age,total_related_records,gap_score,mso_match_status
0,อุดรธานี,เพ็ญ,นาพู่,1,1,0.0,0.0,0,0,NaN,1,1,farmer_only


# แบบฝึกหัดท้ายบท 
ให้ผู้เรียนใช้ table ที่ register ไว้แล้ว ได้แก่ 
- `farmer` 
- `mso` 

เพื่อสร้าง function สำหรับตอบคำถามวิเคราะห์จากสถานการณ์จริง

## แบบฝึกหัดที่ 1: สรุปข้อมูลเกษตรกรรายจังหวัด 

ผู้บริหารต้องการทราบว่าจังหวัดใดมีข้อมูลเกษตรกรเปราะบางมากที่สุด 

และแต่ละจังหวัดมีรายได้และหนี้สินเฉลี่ยเท่าใด ให้สร้าง function ชื่อ

```python 
def summarize_farmer_by_province(con):
    ...
```

ผลลัพธ์ต้องมี column อย่างน้อย

province
- farmer_records
- unique_farmers
- avg_total_income
- avg_total_debt

เงื่อนไข:

- ใช้ COUNT(*) สำหรับจำนวน record
- ใช้ COUNT(DISTINCT pid) สำหรับจำนวนบุคคลไม่ซ้ำ
- ใช้ AVG(total_income) สำหรับรายได้เฉลี่ย
- ใช้ AVG(total_debt) สำหรับหนี้สินเฉลี่ย
- เรียงจาก farmer_records มากไปน้อย

In [1]:
# เขียนคำตอบของคุณใน Cell นี้

## แบบฝึกหัดที่ 2: สรุปข้อมูล MSO รายจังหวัด

ทีมวิเคราะห์ต้องการดูว่าจังหวัดใดมีข้อมูล logbook จาก MSO มากที่สุด และมีจำนวนครัวเรือนไม่ซ้ำเท่าใด

ให้สร้าง function ชื่อ

```python
def summarize_mso_by_province(con):
    ...
```

ผลลัพธ์ต้องมี column อย่างน้อย

- province
- mso_records
- unique_households
- avg_age

เงื่อนไข:

- สรุปข้อมูลระดับจังหวัด
- ใช้ COUNT(*) สำหรับจำนวน record
- ใช้ COUNT(DISTINCT household_id) สำหรับจำนวนครัวเรือนไม่ซ้ำ
- ใช้ AVG(age) สำหรับอายุเฉลี่ย
- เรียงจาก mso_records มากไปน้อย

In [2]:
# เขียนคำตอบของคุณใน Cell นี้

## แบบฝึกหัดที่ 3: เชื่อมข้อมูลระดับจังหวัดด้วย LEFT JOIN

ผู้บริหารต้องการเริ่มจากข้อมูลเกษตรกร และเติมข้อมูล MSO เข้ามา เพื่อดูว่าจังหวัดใดมีข้อมูลจากทั้งสองแหล่ง

ให้สร้าง function ชื่อ

```python
def build_province_left_join_summary(con):
    ...
```

ผลลัพธ์ต้องมี column อย่างน้อย

- province
- farmer_records
- unique_farmers
- avg_total_income
- avg_total_debt
- mso_records
- unique_households
- total_related_records
- mso_match_status

เงื่อนไข:

- ใช้ farmer เป็นข้อมูลหลัก
- ใช้ LEFT JOIN
- ถ้าไม่มีข้อมูลจาก MSO ให้แทน mso_records และ unique_households เป็น 0
- mso_match_status ให้มีค่าเป็น has_mso_data หรือ farmer_only

In [ ]:
# เขียนคำตอบของคุณใน Cell นี้

## แบบฝึกหัดที่ 4: ตรวจสอบ Coverage ของข้อมูลด้วย FULL OUTER JOIN

ทีมข้อมูลต้องการตรวจสอบว่าพื้นที่ใดมีข้อมูลอยู่ในทั้งสองแหล่ง พื้นที่ใดมีเฉพาะ farmer และพื้นที่ใดมีเฉพาะ mso

ให้สร้าง function ชื่อ

```python
def check_area_coverage(con):
    ...
```

ผลลัพธ์ต้องมี column อย่างน้อย

- province
- amphur
- tambon
- farmer_records
- mso_records
- match_status

โดย match_status ต้องแบ่งเป็น

- matched_both
- farmer_only
- mso_only

เงื่อนไข:

- ใช้ FULL OUTER JOIN
- วิเคราะห์ในระดับ province + amphur + tambon
- ใช้ COALESCE เพื่อแสดงชื่อพื้นที่จากฝั่งที่มีข้อมูล

In [3]:
# เขียนคำตอบของคุณใน Cell นี้

## แบบฝึกหัดที่ 5: ค้นหาพื้นที่ที่มี farmer สูงแต่ mso ต่ำ

หน่วยงานต้องการค้นหาพื้นที่ที่อาจควรตรวจสอบเพิ่มเติม 
เพราะมีจำนวน record เกษตรกรสูง แต่มีข้อมูล MSO น้อยหรือไม่มีเลย

ให้สร้าง function ชื่อ

```python
def find_farmer_high_mso_low_area(con, min_farmer_records=10):
    ...
```

ผลลัพธ์ต้องมี column อย่างน้อย

- province
- amphur
- tambon
- farmer_records
- mso_records
- gap_score

กำหนดให้

> gap_score = farmer_records - mso_records

เงื่อนไข:

- ใช้ LEFT JOIN
- วิเคราะห์ในระดับ province + amphur + tambon
- แสดงเฉพาะพื้นที่ที่ farmer_records >= min_farmer_records
- เรียงจาก gap_score มากไปน้อย